# Exploration du flux v2 des stations (issue #22)
Source : flux v2 des prix des carburants 

In [1]:
import sys

import pandas as pd

sys.path.append("..")

from src.cleaning import charger_stations, lire_flux_v2

In [2]:
stations_brutes = lire_flux_v2("../data/raw/flux_v2.csv")

In [3]:
print(stations_brutes.shape)
print(stations_brutes.columns.tolist())
stations_brutes.head()

(9826, 47)
['id', 'latitude', 'longitude', 'Code postal', 'pop', 'Adresse', 'Ville', 'horaires', 'services', 'prix', 'rupture', 'geom', 'Prix Gazole mis à jour le', 'Prix Gazole', 'Prix SP95 mis à jour le', 'Prix SP95', 'Prix E85 mis à jour le', 'Prix E85', 'Prix GPLc mis à jour le', 'Prix GPLc', 'Prix E10 mis à jour le', 'Prix E10', 'Prix SP98 mis à jour le', 'Prix SP98', 'Début rupture e10 (si temporaire)', 'Type rupture e10', 'Début rupture sp98 (si temporaire)', 'Type rupture sp98', 'Début rupture sp95 (si temporaire)', 'Type rupture sp95', 'Début rupture e85 (si temporaire)', 'Type rupture e85', 'Début rupture GPLc (si temporaire)', 'Type rupture GPLc', 'Début rupture gazole (si temporaire)', 'Type rupture gazole', 'Carburants disponibles', 'Carburants indisponibles', 'Carburants en rupture temporaire', 'Carburants en rupture definitive', 'Automate 24-24 (oui/non)', 'Services proposés', 'Département', 'code_departement', 'Région', 'code_region', 'horaires détaillés']


,id,latitude,longitude,Code postal,pop,Adresse,Ville,horaires,services,prix,...,Carburants indisponibles,Carburants en rupture temporaire,Carburants en rupture definitive,Automate 24-24 (oui/non),Services proposés,Département,code_departement,Région,code_region,horaires détaillés
0,34150003,4365100,354700,34150,R,Route de Pézenas,Gignac,"{""@automate-24-24"": ""1"", ""jour"": [{""@id"": ""1"",...","{""service"": [""Bar"", ""Boutique alimentaire"", ""L...","[{""@nom"": ""Gazole"", ""@id"": ""1"", ""@maj"": ""2026-...",...,"SP95,GPLc",NaN,SP95;GPLc,Oui,"Bar,Boutique alimentaire,Lavage automatique,Lo...",Hérault,34,Occitanie,76,"Automate-24-24, Lundi01.00-01.00, Mardi, Mercr..."
1,92230008,4893270,230440,92230,R,192 Avenue Louis Roche,Gennevilliers,NaN,NaN,"[{""@nom"": ""Gazole"", ""@id"": ""1"", ""@maj"": ""2026-...",...,"E85,GPLc",NaN,E85;GPLc,Non,NaN,Hauts-de-Seine,92,Île-de-France,11,NaN
2,25115001,4725741.3021458,592408.72885659,25115,R,Z.A. les Salines,Pouilley-les-Vignes,"{""@automate-24-24"": ""1"", ""jour"": [{""@id"": ""1"",...","{""service"": [""Laverie"", ""DAB (Distributeur aut...","[{""@nom"": ""Gazole"", ""@id"": ""1"", ""@maj"": ""2026-...",...,SP95,NaN,SP95,Oui,"Laverie,DAB (Distributeur automatique de bille...",Doubs,25,Bourgogne-Franche-Comté,27,Automate-24-24
3,40500002,4377300,-56700,40500,R,Route de Mont de Marsan,Saint-Sever,"{""@automate-24-24"": ""1"", ""jour"": [{""@id"": ""1"",...","{""service"": [""Automate CB 24/24"", ""Lavage manu...","[{""@nom"": ""Gazole"", ""@id"": ""1"", ""@maj"": ""2026-...",...,"SP95,E85",NaN,SP95;E85,Oui,"Automate CB 24/24,Lavage manuel,DAB (Distribut...",Landes,40,Nouvelle-Aquitaine,75,Automate-24-24
4,89100001,4818300,330900,89100,R,84 ROUTE DE MAILLOT,Sens,"{""@automate-24-24"": """", ""jour"": [{""@id"": ""1"", ...","{""service"": [""DAB (Distributeur automatique de...","[{""@nom"": ""Gazole"", ""@id"": ""1"", ""@maj"": ""2026-...",...,"SP95,GPLc",NaN,SP95;GPLc,Non,"DAB (Distributeur automatique de billets),Vent...",Yonne,89,Bourgogne-Franche-Comté,27,NaN


In [4]:
print(stations_brutes["pop"].value_counts(dropna=False))
print(stations_brutes["id"].duplicated().sum()) 

pop
R    9388
A     438
Name: count, dtype: int64
0


In [5]:
print(stations_brutes["Région"].value_counts(dropna=False))

sans_region = stations_brutes[stations_brutes["Région"].isna()]
sans_region[["id", "Code postal", "Ville", "code_departement", "pop"]]

Région
Auvergne-Rhône-Alpes          1355
Nouvelle-Aquitaine            1142
Occitanie                     1111
Grand Est                      866
Île-de-France                  831
Hauts-de-France                803
Provence-Alpes-Côte d'Azur     797
Bourgogne-Franche-Comté        668
Normandie                      563
Pays de la Loire               556
Bretagne                       549
Centre-Val de Loire            455
Corse                          126
NaN                              4
Name: count, dtype: int64


,id,Code postal,Ville,code_departement,pop
538,76600022,76600,NaN,NaN,R
1814,56700003,56700,NaN,NaN,R
2572,83150003,83150,NaN,NaN,R
3927,83380002,83380,NaN,NaN,R


In [6]:
stations_idf = charger_stations("../data/raw/flux_v2.csv")
print(stations_idf.shape)

stations_region = stations_brutes[stations_brutes["Région"] == "Île-de-France"]
assert len(stations_idf) == len(stations_region)

(831, 7)


In [7]:
latitude_degres = stations_idf["latitude"].astype(float) / 100000
longitude_degres = stations_idf["longitude"].astype(float) / 100000
print(latitude_degres.min(), latitude_degres.max())   
print(longitude_degres.min(), longitude_degres.max())  

48.145 49.157
1.489 3.361


In [8]:
print(stations_idf["Code postal"].str.len().value_counts())


prefixe_postal = stations_idf["Code postal"].str[:2]
incoherentes = stations_idf[prefixe_postal != stations_idf["code_departement"]]
print(len(incoherentes))
incoherentes[["id", "Ville", "Code postal", "code_departement"]]

Code postal
5    831
Name: count, dtype: int64
4


,id,Ville,Code postal,code_departement
1923,78800001,Bezons,78800,95
2939,92120004,Le Kremlin-Bicêtre,92120,94
4453,94160001,Paris,94160,75
5896,94542003,Paray-Vieille-Poste,94542,91


In [9]:
print((stations_idf["pop"] == "A").sum())  

38


In [10]:
historique_2021 = pd.read_xml(
    "../data/raw/PrixCarburants_annuel_2021.xml",
    xpath=".//pdv",
    attrs_only=True,
    parser="etree",
    encoding="ISO-8859-1",
    dtype=str,
)
print(historique_2021.shape)

(13386, 5)


In [11]:
print(historique_2021["id"].duplicated().sum())  # on attend 0

dans_le_flux = historique_2021["id"].isin(stations_brutes["id"])
absents_du_flux = historique_2021[~dans_le_flux]

print(len(historique_2021), len(absents_du_flux))
print(absents_du_flux["pop"].value_counts())

0
13386 5016
pop
R    4546
A     469
N       1
Name: count, dtype: int64


In [12]:
from src.cleaning import DEPARTEMENTS_IDF, charger_stations, lire_flux_v2

In [13]:
prefixe_cp = historique_2021["cp"].str[:2]
historique_idf = historique_2021[prefixe_cp.isin(DEPARTEMENTS_IDF)]
absents_idf = historique_idf[~historique_idf["id"].isin(stations_brutes["id"])]

print(len(historique_idf), len(absents_idf))
print(absents_idf["pop"].value_counts())


historique_2021[historique_2021["pop"] == "N"]

1607 838
pop
R    798
A     40
Name: count, dtype: int64


,id,latitude,longitude,cp,pop
1516,13500010,4339948,506223,13500,N
13271,95320005,4900805,224185,95320,N
